In [1]:
import pandas as pd

import lsdb


In [2]:
cat_a = lsdb.open_catalog('tests/data/small_sky_order1_collection')

In [3]:
cat_a

,id,ra,dec,ra_error,dec_error
npartitions=4,,,,,
"Order: 1, Pixel: 44",int64[pyarrow],double[pyarrow],double[pyarrow],int64[pyarrow],int64[pyarrow]
"Order: 1, Pixel: 45",...,...,...,...,...
"Order: 1, Pixel: 46",...,...,...,...,...
"Order: 1, Pixel: 47",...,...,...,...,...


In [4]:
def rename_cols(df, names_in, names_out):
    """df = rename_cols(df, ['ra', 'dec'], ['my_ra', 'my_dec'])"""
    for name_in, name_out in zip(names_in, names_out):
        df[name_out] = df[name_in]
    col_names = [col for col in df.columns if col not in names_in]
    return df[col_names]

In [12]:
# Scenario A: Creating an invalid catalog via from_dataframe() / DataFrameCatalogLoader

# Succeeds
cat_b = lsdb.from_dataframe(rename_cols(cat_a.compute(), ['ra', 'dec'], ['my_ra', 'my_dec']), 
    ra_column='my_ra', dec_column='my_dec')

# Fails
# Error happens during catalog creation
cat_b = lsdb.from_dataframe(rename_cols(cat_a.compute(), ['ra', 'dec'], ['my_ra', 'my_dec']))

Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError: No column found for ra

In [13]:
# Scenario B: Invalid catalog via map_partitions()

# This creates an invalid catalog: the hc_structure contains the old ra and dec column names,
# but now there are no columns with those names.
cat_b = cat_a.map_partitions(rename_cols, ['ra', 'dec'], ['my_ra', 'my_dec'])

# But we don't get an error until we try to use the catalog:
cat_a.crossmatch(cat_b)

# Possible fix: make map_partitions() warn when it modifies ra or dec columns

/Users/heather/repos/lsdb/src/lsdb/catalog/catalog.py:410: FutureWarning: The default suffix behavior will change from applying suffixes to all columns to only applying suffixes to overlapping columns in a future release.To maintain the current behavior, explicitly set `suffix_method='all_columns'`. To change to the new behavior, set `suffix_method='overlapping_columns'`.
  warnings.warn(


ValueError: right table must have column ra